# Μέρος 2: Εντοπισμός Χωρο-χρονικών Σημείων Ενδιαφέροντος και Εξαγωγή Χαρακτηριστικών σε Βίντεο Ανθρωπίνων Δράσεων

In [21]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from scipy import ndimage
from part2_data import cv24_lab2_2_utils as utils

# read video
video = utils.read_video('part2_data/boxing/person02_boxing_d3_uncomp.avi', 200, 0)
video.shape

(120, 160, 200)

## 2.1 Χωρο-χρονικά Σημεία Ενδιαφέροντος

In [22]:
def HarrisDetector(video, s, sigma, tau, k=0.005, theta_corn=0.2):
    video = video.copy()

    # Get Gaussian kernel for video
    space_size = int(2*np.ceil(3*sigma)+1)
    time_size = int(2*np.ceil(3*tau)+1)
    space_kernel = cv2.getGaussianKernel(space_size, sigma).T[0]
    time_kernel = cv2.getGaussianKernel(time_size, tau).T[0]
    
    # Normalize and smoothen video
    video = video.astype(float)/video.max()
    video = ndimage.convolve1d(video, space_kernel, axis=0)
    video = ndimage.convolve1d(video, space_kernel, axis=1)
    video = ndimage.convolve1d(video, time_kernel, axis=2)

    # Calculate gradients
    Ly = ndimage.convolve1d(video, np.array([-1, 0, 1]), axis=0)
    Lx = ndimage.convolve1d(video, np.array([-1, 0, 1]), axis=1)
    Lt = ndimage.convolve1d(video, np.array([-1, 0, 1]), axis=2)

    # Get Gaussian kernel for gradient
    space_size = int(2*np.ceil(3*s*sigma)+1)
    time_size = int(2*np.ceil(3*s*tau)+1)
    space_kernel = cv2.getGaussianKernel(space_size, s*sigma).T[0]
    time_kernel = cv2.getGaussianKernel(time_size, s*tau).T[0]

    # Smoothen the gradient products
    Lxy = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lx * Ly, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lxt = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lx * Lt, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lyt = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Ly * Lt, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lxx = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lx * Lx, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Lyy = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Ly * Ly, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)
    Ltt = ndimage.convolve1d(ndimage.convolve1d(ndimage.convolve1d(Lt * Lt, space_kernel, axis=0), space_kernel, axis=1), time_kernel, axis=2)

    # Harris cornerness criterion
    trace = Lxx + Lyy + Ltt
    det = Lxx*(Lyy*Ltt - Lyt*Lyt) - Lxy*(Lxy*Ltt - Lyt*Lxt) + Lxt*(Lxy*Lyt - Lyy*Lxt)
    H = abs(det - k * trace**3)
    H = np.array(H)

    # # Apply threshold (condition Σ2)
    H[H < theta_corn*H.max()] = 0

    # Return 500 points with highest cornerness
    y, x, t = np.unravel_index(np.argsort(-1*H, axis=None), H.shape)
    scale = sigma*np.ones(x.shape)
    points = list(zip(x,y,t,scale))
    return np.array(points[:500])

# Test function
harris_points = HarrisDetector(video=video, s=2, sigma=4, tau=1.5)
utils.show_detection(video, harris_points)

(500, 4)

In [26]:
def GaborDetector(video, sigma, tau, theta_corn=0.02):
    video = video.copy()

    # Get Gaussian kernel for video
    space_size = int(2*np.ceil(3*sigma)+1)
    kernel = cv2.getGaussianKernel(space_size, sigma).T[0]

    # Normalize and smoothen video
    video = video.astype(float)/video.max()
    video = ndimage.convolve1d(video, kernel, axis=0)
    video = ndimage.convolve1d(video, kernel, axis=1)

    # Define Gabor filters
    t = np.linspace(-2*tau, 2*tau, int(4*tau+1))
    omega = 4/tau
    h_ev = np.cos(2*np.pi*omega*t)*np.exp(-t**2/(2*tau**2))
    h_od = np.sin(2*np.pi*omega*t)*np.exp(-t**2/(2*tau**2))

    # Normalize with L1 norm
    h_ev /= np.linalg.norm(h_ev, ord=1)
    h_od /= np.linalg.norm(h_od, ord=1)

    # Gabor cornerness criterion
    H = (ndimage.convolve1d(video, h_ev, axis=2)**2) + (ndimage.convolve1d(video, h_od, axis=2)**2)
    
    # Apply threshold (condition Σ2)
    H[H < theta_corn*H.max()] = 0

    # Return 500 points with highest cornerness
    y, x, t = np.unravel_index(np.argsort(-1*H, axis=None), H.shape)
    scale = sigma*np.ones(x.shape)
    points = list(zip(x,y,t,scale))
    return np.array(points[:500])

# Test function
gabor_points = GaborDetector(video=video, sigma=2, tau=1.5, theta_corn=0.2)
utils.show_detection(video, gabor_points)

## 2.2 Χωρο-χρονικοί Ιστογραφικοί Περιγραφητές

## 2.3: Κατασκευή Bag of Visual Words και χρήση Support Vector Machines για την ταξινόμηση δράσεων